<a href="https://colab.research.google.com/github/busybee-123/Pollinator_Cam/blob/main/2.%20Processing%20training%20dataset/Segment_and_create_composite_images.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Segment dataset and create composite images

This code assumes that images have been downloaded from GBIF and organised into a folder called `dataset`.
It also assumes that the classes to use in initial labelling for YOLO training correspond to folders in a folder called `GBIF images`.

## Configuration

In [ ]:
# Mount Google Drive to access your files
from google.colab import drive
drive.mount('/content/drive')

# Install Pillow for image processing
%pip install Pillow

# Clone and install flat-bug if not already present
import sys
import os
if not os.path.exists('flat-bug'):
    !git clone https://github.com/darsa-group/flat-bug.git
!bash -c "cd flat-bug && {sys.executable} -m pip install -e ."

## Prepare new training dataset

### Run Flatbug on the entire original dataset to get segmented crops

In [ ]:
!fb_predict --no-overviews -R -i /content/drive/MyDrive/Colab\ Notebooks/Pollinator\ Camera\ Project/dataset -o /content/drive/MyDrive/Colab\ Notebooks/Pollinator\ Camera\ Project/dataset_flatbug_crops

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
YOLOv8m-seg summary (fused): 111 layers, 24,586,035 parameters, 0 gradients, 93.0 GFLOPs
Processing images: 100% 2760/2760 [18:21<00:00,  2.51image/s, Processing cropped_Yellow_faced_bee__Hylaeus__5939007084__supplementary_preserved.jpg]
Finishing pending executions.: 100% 1/1 [00:00<00:00, 878.02it/s]


### Find how many images Flatbug successfully ran on

In [ ]:
import os

dataset_flatbug_crops_dir = '/content/drive/MyDrive/Colab Notebooks/Pollinator Camera Project/dataset_flatbug_crops'

image_extensions = ('.png', '.jpg', '.jpeg', '.gif', '.bmp')

# Initialize counters for each category
human_category = {'total_folders': 0, '0_images': 0, '1_image': 0, '2_plus_images': 0}
preserved_category = {'total_folders': 0, '0_images': 0, '1_image': 0, '2_plus_images': 0}
other_category = {'total_folders': 0, '0_images': 0, '1_image': 0, '2_plus_images': 0}

# Walk through the dataset_flatbug_crops directory
for root, dirs, files in os.walk(dataset_flatbug_crops_dir):
    # We are interested in immediate subdirectories that contain image files
    # The flatbug output structure creates a subdirectory for each input image
    # where the cropped images are placed.
    if root == dataset_flatbug_crops_dir:
        continue # Skip the root directory itself, only process its immediate subdirectories

    current_folder_image_count = 0
    for file in files:
        if file.lower().endswith(image_extensions):
            current_folder_image_count += 1

    # Determine category based on folder name
    folder_category = None
    if 'human' in root.lower():
        folder_category = human_category
    elif 'preserved' in root.lower():
        folder_category = preserved_category
    else:
        folder_category = other_category

    # Update counts for the determined category
    folder_category['total_folders'] += 1
    if current_folder_image_count == 0:
        folder_category['0_images'] += 1
    elif current_folder_image_count == 1:
        folder_category['1_image'] += 1
    else:
        folder_category['2_plus_images'] += 1

# Function to print results for a category
def print_category_stats(name, category_data):
    print(f"\n--- Category: {name} ---")
    total = category_data['total_folders']
    if total == 0:
        print("No folders found in this category.")
        return

    print(f"Total folders: {total}")
    print(f"  Folders with 0 images: {category_data['0_images']} ({category_data['0_images'] / total:.2%})")
    print(f"  Folders with 1 image: {category_data['1_image']} ({category_data['1_image'] / total:.2%})")
    print(f"  Folders with 2+ images: {category_data['2_plus_images']} ({category_data['2_plus_images'] / total:.2%})")


# Print results for each category
print_category_stats("Contains 'human' in folder name", human_category)
print_category_stats("Contains 'preserved' in folder name", preserved_category)
print_category_stats("Does not contain 'human' or 'preserved'", other_category)


### Delete folders that do not contain exactly 1 image

In [ ]:
import os
import shutil

dataset_flatbug_crops_dir = '/content/drive/MyDrive/Colab Notebooks/Pollinator Camera Project/dataset_flatbug_crops'
image_extensions = ('.png', '.jpg', '.jpeg', '.gif', '.bmp')

# List to store directories to delete
dirs_to_delete = []

for root, dirs, files in os.walk(dataset_flatbug_crops_dir):
    # We are looking for 'crops' subdirectories
    if os.path.basename(root) == 'crops':
        current_folder_image_count = 0
        for file in files:
            if file.lower().endswith(image_extensions):
                current_folder_image_count += 1

        if current_folder_image_count != 1:
            # If it doesn't contain exactly 1 image, mark its parent for deletion
            parent_dir = os.path.dirname(root)
            dirs_to_delete.append(parent_dir)

# Delete the marked directories (from most specific to most general)
# To avoid issues with deleting a parent then trying to delete its child
dirs_to_delete = sorted(list(set(dirs_to_delete)), key=len, reverse=True)

for d in dirs_to_delete:
    if os.path.exists(d):
        print(f"Deleting: {d} (does not contain exactly 1 image in its 'crops' subdirectory).")
        shutil.rmtree(d)
    else:
        print(f"Skipping deletion: {d} (already removed or not found).")

print("Cleanup complete.")

In [ ]:
import os
import shutil
import random

dataset_flatbug_crops_dir = '/content/drive/MyDrive/Colab Notebooks/Pollinator Camera Project/dataset_flatbug_crops'
output_base_dir = '/content/drive/MyDrive/Colab Notebooks/Pollinator Camera Project/dataset_split'

# Define the target directories
multi_dir = os.path.join(output_base_dir, 'multi')
single_dir = os.path.join(output_base_dir, 'single')

# Create target directories if they don't exist
os.makedirs(multi_dir, exist_ok=True)
os.makedirs(single_dir, exist_ok=True)

# Get all immediate subdirectories in dataset_flatbug_crops_dir
subdirectories = [d for d in os.listdir(dataset_flatbug_crops_dir) if os.path.isdir(os.path.join(dataset_flatbug_crops_dir, d))]

# Shuffle the list of subdirectories to ensure a random split
random.shuffle(subdirectories)

# Calculate the split point for roughly equal halves
split_point = len(subdirectories) // 2

# Assign directories to 'multi' and 'single' groups
multi_group = subdirectories[:split_point]
single_group = subdirectories[split_point:]

print(f"Splitting {len(subdirectories)} folders: {len(multi_group)} for 'multi', {len(single_group)} for 'single'.")

# Move folders to the 'multi' directory
for folder_name in multi_group:
    src_path = os.path.join(dataset_flatbug_crops_dir, folder_name)
    dst_path = os.path.join(multi_dir, folder_name)
    try:
        shutil.move(src_path, dst_path)
        print(f"Moved {folder_name} to {multi_dir}")
    except Exception as e:
        print(f"Error moving {folder_name} to {multi_dir}: {e}")

# Move folders to the 'single' directory
for folder_name in single_group:
    src_path = os.path.join(dataset_flatbug_crops_dir, folder_name)
    dst_path = os.path.join(single_dir, folder_name)
    try:
        shutil.move(src_path, dst_path)
        print(f"Moved {folder_name} to {single_dir}")
    except Exception as e:
        print(f"Error moving {folder_name} to {single_dir}: {e}")

print("Dataset split complete.")

### Define categories

In [ ]:
import os

base_gbif_images_path = '/content/drive/MyDrive/Colab Notebooks/Pollinator Camera Project/GBIF images'

print(f"Listing immediate subfolders in: {base_gbif_images_path}\n")

if os.path.exists(base_gbif_images_path):
    all_categories = [
        d for d in os.listdir(base_gbif_images_path)
        if os.path.isdir(os.path.join(base_gbif_images_path, d))
    ]
    all_categories.sort() # Sort alphabetically for consistent numbering

    if all_categories:
        for i, category in enumerate(all_categories):
            print(f"{i}: {category}")
    else:
        print("No subfolders found in the 'GBIF images' directory.")
else:
    print(f"Error: The path '{base_gbif_images_path}' does not exist.")


Listing immediate subfolders in: /content/drive/MyDrive/Colab Notebooks/Pollinator Camera Project/GBIF images

0: Ant
1: Aphid_eater_hoverfly
2: Batman_hoverfly
3: Beetle
4: Common_wasp
5: Drone_fly
6: Flower_bee
7: Ginger_yellow_bumblebee
8: Globetail_hoverfly
9: Honeybee
10: Humming_Syrphus
11: Large_white
12: Leafcutter_bee
13: Marmalade_hoverfly
14: Mason_bee
15: Mint_moth
16: Narcissus_bulb_fly
17: Orange_tailed_mining_bee
18: Other_arthropods
19: Other_fly
20: Other_moth_or_butterfly
21: Red_tailed_bumblebee
22: Small_scissor_bee
23: Small_white
24: Speckled_wood
25: Sweat_bee
26: Thick_legged_hoverfly
27: White_tailed_bumblebee
28: Wool_carder_bee
29: Yellow_faced_bee


### Superimpose segmented crops on background image to create composite images and generate YOLO annotations

In [ ]:
from PIL import Image
import os
import random
import re

# Assuming all_categories and category_to_id are defined globally from previous cells.
# They are needed for extract_category_from_filename.

# Define helper function (copied for self-containment)
def clean_filename_for_output(filename):
    cleaned = re.sub(r'_CROPNUMBER_\d+_UUID_[0-9a-fA-F-]+', '', filename)
    cleaned = re.sub(r'_CROPNUMBER_\d+$', '', cleaned)
    cleaned = re.sub(r'_UUID_[0-9a-fA-F-]+$', '', cleaned)
    cleaned = re.sub(r'^(synthetic_)?crop_cropped_|^crop_|^cropped_', '', cleaned)
    return cleaned

# Define helper function (copied for self-containment)
def extract_category_from_filename(filename):
    # Remove the 'synthetic_' prefix
    name_without_prefix = filename.replace('synthetic_', '', 1)

    # Remove the '.txt' extension
    name_without_ext = os.path.splitext(name_without_prefix)[0]

    # Check if any category name from all_categories is present in the filename
    # Sort by length in descending order to ensure longer names are matched first
    sorted_categories = sorted(all_categories, key=len, reverse=True)
    for category in sorted_categories:
        if category in name_without_ext:
            return category

    # print(f"Warning: Could not reliably extract category from {filename}. Manual inspection might be needed.")
    return None

source_insect_dir = '/content/drive/MyDrive/Colab Notebooks/Pollinator Camera Project/dataset'
background_image_path = '/content/drive/MyDrive/Colab Notebooks/Pollinator Camera Project/frame.jpg'

# New output directories to distinguish from previous runs
output_images_dir = '/content/drive/MyDrive/Colab Notebooks/Pollinator Camera Project/synthetic_dataset/images'
output_labels_dir = '/content/drive/MyDrive/Colab Notebooks/Pollinator Camera Project/synthetic_dataset/labels'

# Create output directories if they don't exist
os.makedirs(output_images_dir, exist_ok=True)
os.makedirs(output_labels_dir, exist_ok=True)

# Load the background image
try:
    background = Image.open(background_image_path).convert("RGB")
    bg_width, bg_height = background.size
except FileNotFoundError:
    print(f"Error: Background image not found at {background_image_path}")
    exit()
except Exception as e:
    print(f"Error loading background image: {e}")
    exit()

# Get all subdirectories (which are the insect folders) in the source_insect_dir
insect_folders = [d for d in os.listdir(source_insect_dir) if os.path.isdir(os.path.join(source_insect_dir, d))]

if not insect_folders:
    print(f"No insect folders found in {source_insect_dir}")
else:
    print(f"Found {len(insect_folders)} insect folders to process.")

for folder_name in insect_folders:
    folder_path = os.path.join(source_insect_dir, folder_name)
    # The actual image is expected inside a 'crops' subfolder within each insect folder
    crops_folder_path = os.path.join(folder_path, 'crops')

    png_files_in_crop = [f for f in os.listdir(crops_folder_path) if f.lower().endswith('.png')]

    if not png_files_in_crop:
        print(f"No PNG files found in {crops_folder_path}. Skipping.")
        continue

    # Assuming there's exactly one PNG per crops folder after the previous cleanup
    png_filename = png_files_in_crop[0]
    png_filepath = os.path.join(crops_folder_path, png_filename)

    try:
        insect_img = Image.open(png_filepath).convert("RGBA")

        # Determine target size: 5-15% of the *smaller* background dimension
        min_bg_dim = min(bg_width, bg_height)
        scale_factor = random.uniform(0.05, 0.15)
        target_side_length = int(min_bg_dim * scale_factor)

        # Resize insect image while maintaining aspect ratio
        original_insect_width, original_insect_height = insect_img.size
        if original_insect_width == 0 or original_insect_height == 0:
            print(f"Warning: Insect image {png_filename} has zero dimension. Skipping.")
            continue

        if original_insect_width > original_insect_height:
            new_insect_width = target_side_length
            new_insect_height = int(original_insect_height * (target_side_length / original_insect_width))
        else:
            new_insect_height = target_side_length
            new_insect_width = int(original_insect_width * (target_side_length / original_insect_height))

        # Ensure new dimensions are at least 1x1
        new_insect_width = max(1, new_insect_width)
        new_insect_height = max(1, new_insect_height)

        resized_insect = insect_img.resize((new_insect_width, new_insect_height), Image.LANCZOS)

        # Calculate random position to place the insect fully within the background
        max_x = bg_width - new_insect_width
        max_y = bg_height - new_insect_height

        if max_x < 0 or max_y < 0:
            print(f"Warning: Insect image {png_filename} is larger than background even after scaling ({new_insect_width}x{new_insect_height} vs {bg_width}x{bg_height}). Skipping.")
            continue

        paste_x = random.randint(0, max_x)
        paste_y = random.randint(0, max_y)

        # Create a copy of the background to paste on for each insect
        combined_img = background.copy()

        # Paste the resized insect image onto the background using its alpha channel as a mask
        combined_img.paste(resized_insect, (paste_x, paste_y), resized_insect)

        # Clean filename for output
        cleaned_filename = clean_filename_for_output(os.path.splitext(png_filename)[0])

        # Save the combined image
        output_img_filename = f"synthetic_{cleaned_filename}.jpg" # Changed prefix
        output_img_filepath = os.path.join(output_images_dir, output_img_filename)
        combined_img.save(output_img_filepath)

        # Generate YOLO annotation file
        # Calculate normalized bounding box coordinates
        bbox_x_center = (paste_x + new_insect_width / 2) / bg_width
        bbox_y_center = (paste_y + new_insect_height / 2) / bg_height
        bbox_width = new_insect_width / bg_width
        bbox_height = new_insect_height / bg_height

        # Extract category and get ID using the helper function
        category_name = extract_category_from_filename(os.path.basename(png_filepath))
        category_id = category_to_id.get(category_name, 0) # Default to 0 if not found

        yolo_annotation = f"{category_id} {bbox_x_center} {bbox_y_center} {bbox_width} {bbox_height}"
        output_label_filename = f"synthetic_{cleaned_filename}.txt" # Changed prefix
        output_label_filepath = os.path.join(output_labels_dir, output_label_filename)

        with open(output_label_filepath, 'w') as f:
            f.write(yolo_annotation)

        print(f"Processed {png_filename}: Saved image to {output_images_dir} and annotation to {output_labels_dir}")

    except Exception as e:
        print(f"Error processing {png_filename}: {e}")

print("Finished generating synthetic dataset with YOLO annotations.")

### Inspect images for sanity check

In [ ]:
from PIL import Image, ImageDraw
import os
import random
import matplotlib.pyplot as plt

# Assuming category_to_id is defined globally from previous cells.
# Reverse mapping from ID to category name for display
id_to_category = {v: k for k, v in category_to_id.items()}

def plot_image_with_bboxes(image_path, label_path, dataset_type):
    try:
        img = Image.open(image_path).convert("RGB")
        draw = ImageDraw.Draw(img)
        img_width, img_height = img.size

        with open(label_path, 'r') as f:
            for line in f:
                parts = line.strip().split(' ')
                if len(parts) == 5:
                    class_id = int(parts[0])
                    x_center, y_center, bbox_width, bbox_height = map(float, parts[1:])

                    # Convert normalized YOLO coordinates to pixel coordinates
                    x_min = int((x_center - bbox_width / 2) * img_width)
                    y_min = int((y_center - bbox_height / 2) * img_height)
                    x_max = int((x_center + bbox_width / 2) * img_width)
                    y_max = int((y_center + bbox_height / 2) * img_height)

                    # Draw bounding box (red rectangle)
                    draw.rectangle([x_min, y_min, x_max, y_max], outline="red", width=3)

                    # Get category name for display
                    category_name = id_to_category.get(class_id, f"Unknown_{class_id}")
                    draw.text((x_min + 5, y_min + 5), category_name, fill="red")

        plt.imshow(img)
        plt.title(f"{dataset_type}: {os.path.basename(image_path)}")
        plt.axis('off')
        plt.show()

    except FileNotFoundError:
        print(f"Error: File not found - {image_path} or {label_path}")
    except Exception as e:
        print(f"Error processing {image_path}: {e}")


# --- Display 3 synthetic frames  ---
print("Displaying 3 synthetic frames:")
output_images_dir_= '/content/drive/MyDrive/Colab Notebooks/Pollinator Camera Project/synthetic_dataset/images'
output_labels_dir = '/content/drive/MyDrive/Colab Notebooks/Pollinator Camera Project/synthetic_dataset/labels'

images = [f for f in os.listdir(output_images_dir) if f.endswith('.jpg')]
random.shuffle(images)

for i in range(min(3, len(images))):
    img_filename = images[i]
    label_filename = img_filename.replace('.jpg', '.txt')
    image_path = os.path.join(output_images_dir, img_filename)
    label_path = os.path.join(output_labels_dir, label_filename)
    plot_image_with_bboxes(image_path, label_path)